# Model Comparison — Energy Demand (t+1)

## Objective

Evaluate how model complexity affects prediction performance for next-day energy demand.

We compare three model classes:

- Linear Regression (simple baseline)
- Random Forest (nonlinear model)
- Gradient Boosting (higher flexibility)

Across two feature settings:

1. Without weather
2. With weather

---

## Key Question

Does increasing model complexity improve prediction performance?

Or does adding information (weather) reduce the need for complex models?

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

from configs.config import *

In [2]:
file_path = PROCESSED_DATA_DIR / "aep_features.csv"

df_model = pd.read_csv(file_path)
df_model["Datetime"] = pd.to_datetime(df_model["Datetime"])

df_model.head()

,Datetime,AEP_MW,temp_mean,load_lag_1,load_lag_2,load_lag_3,load_lag_7,day_of_week,is_weekend,target
0,2004-10-08,14350.333333,16.0,14449.416667,14424.791667,14439.708333,14284.521739,4,0,12934.541667
1,2004-10-09,12934.541667,17.2,14350.333333,14449.416667,14424.791667,12999.875000,5,1,12260.375000
2,2004-10-10,12260.375000,12.5,12934.541667,14350.333333,14449.416667,12227.083333,6,1,14299.750000
3,2004-10-11,14299.750000,11.3,12260.375000,12934.541667,14350.333333,14309.041667,0,0,14614.916667
4,2004-10-12,14614.916667,11.4,14299.750000,12260.375000,12934.541667,14439.708333,1,0,14454.416667


In [3]:
df_model.columns

Index(['Datetime', 'AEP_MW', 'temp_mean', 'load_lag_1', 'load_lag_2',
       'load_lag_3', 'load_lag_7', 'day_of_week', 'is_weekend', 'target'],
      dtype='str')

In [4]:
features_no_weather = [
    "load_lag_1",
    "load_lag_2",
    "load_lag_3",
    "load_lag_7",
    "day_of_week",
    "is_weekend"
]

features_with_weather = features_no_weather + ["temp_mean"]

In [5]:
X_no = df_model[features_no_weather]
X_w  = df_model[features_with_weather]

y = df_model["target"]

In [6]:
split_idx = int(len(df_model) * 0.8)

X_train_no = X_no.iloc[:split_idx]
X_test_no  = X_no.iloc[split_idx:]

X_train_w = X_w.iloc[:split_idx]
X_test_w  = X_w.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

In [7]:
print("Train shape:", X_train_no.shape)
print("Test shape:", X_test_no.shape)

Train shape: (4037, 6)
Test shape: (1010, 6)


In [8]:
lr = LinearRegression()

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=RANDOM_STATE
)

In [9]:
lr_no = lr.fit(X_train_no, y_train)
rf_no = rf.fit(X_train_no, y_train)
gb_no = gb.fit(X_train_no, y_train)

In [10]:
lr_w = LinearRegression().fit(X_train_w, y_train)
rf_w = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
).fit(X_train_w, y_train)

gb_w = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=RANDOM_STATE
).fit(X_train_w, y_train)

In [11]:
# No weather
pred_lr_no = lr_no.predict(X_test_no)
pred_rf_no = rf_no.predict(X_test_no)
pred_gb_no = gb_no.predict(X_test_no)

# With weather
pred_lr_w = lr_w.predict(X_test_w)
pred_rf_w = rf_w.predict(X_test_w)
pred_gb_w = gb_w.predict(X_test_w)

In [12]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [13]:
results = pd.DataFrame({
    "Model": ["Linear", "Random Forest", "Gradient Boosting"],
    
    "No Weather": [
        rmse(y_test, pred_lr_no),
        rmse(y_test, pred_rf_no),
        rmse(y_test, pred_gb_no)
    ],
    
    "With Weather": [
        rmse(y_test, pred_lr_w),
        rmse(y_test, pred_rf_w),
        rmse(y_test, pred_gb_w)
    ]
})

results

,Model,No Weather,With Weather
0,Linear,1335.000423,1325.075038
1,Random Forest,1176.702451,977.778022
2,Gradient Boosting,1141.587730,965.253986


In [14]:
df_model[["temp_mean"]].describe()

,temp_mean
count,5047.000000
mean,11.634535
std,10.458565
min,-22.900000
25%,3.200000
50%,12.600000
75%,20.900000
max,32.100000


In [15]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [17]:
rmse_table = pd.DataFrame({
    "Model": ["Linear", "Random Forest", "Gradient Boosting"],
    "No Weather": [
        rmse(y_test, pred_lr_no),
        rmse(y_test, pred_rf_no),
        rmse(y_test, pred_gb_no)
    ],
    "With Weather": [
        rmse(y_test, pred_lr_w),
        rmse(y_test, pred_rf_w),
        rmse(y_test, pred_gb_w)
    ]
})

rmse_table

,Model,No Weather,With Weather
0,Linear,1335.000423,1325.075038
1,Random Forest,1176.702451,977.778022
2,Gradient Boosting,1141.587730,965.253986


In [18]:
mae_table = pd.DataFrame({
    "Model": ["Linear", "Random Forest", "Gradient Boosting"],
    "No Weather": [
        mae(y_test, pred_lr_no),
        mae(y_test, pred_rf_no),
        mae(y_test, pred_gb_no)
    ],
    "With Weather": [
        mae(y_test, pred_lr_w),
        mae(y_test, pred_rf_w),
        mae(y_test, pred_gb_w)
    ]
})

mae_table

,Model,No Weather,With Weather
0,Linear,1058.822140,1061.269496
1,Random Forest,916.232061,763.617697
2,Gradient Boosting,897.117696,760.547466


In [19]:
mape_table = pd.DataFrame({
    "Model": ["Linear", "Random Forest", "Gradient Boosting"],
    "No Weather": [
        mape(y_test, pred_lr_no),
        mape(y_test, pred_rf_no),
        mape(y_test, pred_gb_no)
    ],
    "With Weather": [
        mape(y_test, pred_lr_w),
        mape(y_test, pred_rf_w),
        mape(y_test, pred_gb_w)
    ]
})

mape_table

,Model,No Weather,With Weather
0,Linear,7.295783,7.333819
1,Random Forest,6.216358,5.275965
2,Gradient Boosting,6.109497,5.285785


## Linear Model Extension — Adding a Quadratic Temperature Term

The scatter plot of next-day load versus temperature suggests a U-shaped relationship.

To test whether the weak performance of the linear model is due to an overly simple temperature representation, we add a quadratic temperature feature:

- `temp_mean`
- `temp_sq = temp_mean^2`

and retrain the linear model.

In [20]:
df_model["temp_sq"] = df_model["temp_mean"] ** 2

df_model[["temp_mean", "temp_sq"]].head()

,temp_mean,temp_sq
0,16.0,256.00
1,17.2,295.84
2,12.5,156.25
3,11.3,127.69
4,11.4,129.96


In [21]:
features_linear_quad = [
    "load_lag_1",
    "load_lag_2",
    "load_lag_3",
    "load_lag_7",
    "day_of_week",
    "is_weekend",
    "temp_mean",
    "temp_sq"
]

In [22]:
X_quad = df_model[features_linear_quad]
y_quad = df_model["target"]

X_train_quad = X_quad.iloc[:split_idx]
X_test_quad  = X_quad.iloc[split_idx:]

y_train_quad = y_quad.iloc[:split_idx]
y_test_quad  = y_quad.iloc[split_idx:]

In [23]:
lr_quad = LinearRegression()
lr_quad.fit(X_train_quad, y_train_quad)

pred_lr_quad = lr_quad.predict(X_test_quad)

In [24]:
linear_extension_table = pd.DataFrame({
    "Linear Model Variant": [
        "Temp only",
        "Temp + Temp^2"
    ],
    "RMSE": [
        rmse(y_test, pred_lr_w),
        rmse(y_test_quad, pred_lr_quad)
    ],
    "MAE": [
        mae(y_test, pred_lr_w),
        mae(y_test_quad, pred_lr_quad)
    ],
    "MAPE %": [
        mape(y_test, pred_lr_w),
        mape(y_test_quad, pred_lr_quad)
    ]
})

linear_extension_table.round(2)

,Linear Model Variant,RMSE,MAE,MAPE %
0,Temp only,1325.08,1061.27,7.33
1,Temp + Temp^2,1154.94,933.91,6.59


In [34]:
linear_extension_table = pd.DataFrame({
    "Linear Model Variant": [
        "Temp only",
        "Temp + Temp^2"
    ],
    "MAPE %": [
        mape(y_test, pred_lr_w),
        mape(y_test_quad, pred_lr_quad)
    ]
})

linear_extension_table.round(2)

,Linear Model Variant,MAPE %
0,Temp only,7.33
1,Temp + Temp^2,6.59


In [25]:
df_model["hdd"] = np.maximum(18 - df_model["temp_mean"], 0)
df_model["cdd"] = np.maximum(df_model["temp_mean"] - 18, 0)

In [26]:
features_linear_hdd = [
    "load_lag_1",
    "load_lag_2",
    "load_lag_3",
    "load_lag_7",
    "day_of_week",
    "is_weekend",
    "hdd",
    "cdd"
]

In [27]:
X_hdd = df_model[features_linear_hdd]
y_hdd = df_model["target"]

X_train_hdd = X_hdd.iloc[:split_idx]
X_test_hdd  = X_hdd.iloc[split_idx:]

y_train_hdd = y_hdd.iloc[:split_idx]
y_test_hdd  = y_hdd.iloc[split_idx:]

In [28]:
lr_hdd = LinearRegression()
lr_hdd.fit(X_train_hdd, y_train_hdd)

pred_lr_hdd = lr_hdd.predict(X_test_hdd)

In [29]:
linear_comparison_full = pd.DataFrame({
    "Linear Model Variant": [
        "Temp only",
        "Temp + Temp^2",
        "HDD + CDD"
    ],
    "RMSE": [
        rmse(y_test, pred_lr_w),
        rmse(y_test_quad, pred_lr_quad),
        rmse(y_test_hdd, pred_lr_hdd)
    ],
    "MAE": [
        mae(y_test, pred_lr_w),
        mae(y_test_quad, pred_lr_quad),
        mae(y_test_hdd, pred_lr_hdd)
    ],
    "MAPE %": [
        mape(y_test, pred_lr_w),
        mape(y_test_quad, pred_lr_quad),
        mape(y_test_hdd, pred_lr_hdd)
    ]
})

linear_comparison_full.round(2)

,Linear Model Variant,RMSE,MAE,MAPE %
0,Temp only,1325.08,1061.27,7.33
1,Temp + Temp^2,1154.94,933.91,6.59
2,HDD + CDD,1175.79,949.00,6.73


In [30]:
features_linear_full = [
    "load_lag_1",
    "load_lag_2",
    "load_lag_3",
    "load_lag_7",
    "day_of_week",
    "is_weekend",
    "temp_mean",
    "temp_sq",
    "hdd",
    "cdd"
]

In [31]:
X_full = df_model[features_linear_full]
y_full = df_model["target"]

X_train_full = X_full.iloc[:split_idx]
X_test_full  = X_full.iloc[split_idx:]

y_train_full = y_full.iloc[:split_idx]
y_test_full  = y_full.iloc[split_idx:]

In [32]:
lr_full = LinearRegression()
lr_full.fit(X_train_full, y_train_full)

pred_lr_full = lr_full.predict(X_test_full)

In [33]:
linear_comparison_full2 = pd.DataFrame({
    "Linear Model Variant": [
        "Temp only",
        "Temp + Temp^2",
        "HDD + CDD",
        "Temp + Temp^2 + HDD + CDD"
    ],
    "RMSE": [
        rmse(y_test, pred_lr_w),
        rmse(y_test_quad, pred_lr_quad),
        rmse(y_test_hdd, pred_lr_hdd),
        rmse(y_test_full, pred_lr_full)
    ],
    "MAE": [
        mae(y_test, pred_lr_w),
        mae(y_test_quad, pred_lr_quad),
        mae(y_test_hdd, pred_lr_hdd),
        mae(y_test_full, pred_lr_full)
    ],
    "MAPE %": [
        mape(y_test, pred_lr_w),
        mape(y_test_quad, pred_lr_quad),
        mape(y_test_hdd, pred_lr_hdd),
        mape(y_test_full, pred_lr_full)
    ]
})

linear_comparison_full2.round(2)

,Linear Model Variant,RMSE,MAE,MAPE %
0,Temp only,1325.08,1061.27,7.33
1,Temp + Temp^2,1154.94,933.91,6.59
2,HDD + CDD,1175.79,949.00,6.73
3,Temp + Temp^2 + HDD + CDD,1141.75,920.70,6.55


## Final Insight — Model vs Representation vs System Structure

This experiment began as a comparison between linear and nonlinear models for next-day energy demand prediction. However, the results point to a deeper conclusion about the relationship between model complexity, feature representation, and underlying system structure.

### Key Observations

- A basic linear model using temperature performed poorly (~7.3% MAPE).
- Adding a quadratic temperature term (`temp²`) significantly improved performance (~6.6% MAPE), capturing the dominant U-shaped relationship between temperature and demand.
- Threshold-based features (HDD/CDD) provided only partial improvement, suggesting the system is not well described by rigid piecewise linear behavior.
- Combining multiple temperature representations yielded only marginal gains (~6.55% MAPE), indicating diminishing returns from global feature engineering.
- Nonlinear models (Random Forest, Gradient Boosting) still outperformed all linear variants (~5.2% MAPE).

### Structural Insight

The temperature–demand relationship is not only nonlinear, but also asymmetric and context-dependent:

- Demand increases in both cold and hot conditions, but with different slopes and curvature.
- This asymmetry cannot be fully captured by simple global transformations such as quadratic or piecewise linear features.
- Linear models, even with feature expansion, assume a single global functional form and struggle to represent such structure.

In contrast, tree-based models implicitly learn:
- different behaviors across temperature ranges,
- local variations,
- and interactions between temperature, lagged demand, and time features.

### Conclusion

The primary limitation of the linear model was not the absence of temperature information, but the inability to represent the asymmetric and locally varying structure of the system.

While feature engineering (e.g., `temp²`) captures the dominant nonlinearity, a residual performance gap remains, suggesting that more flexible models are required to model interactions and localized effects.

This highlights a broader principle:

> Improving feature representation can significantly reduce the need for model complexity — but only up to the point where the underlying system structure becomes too rich to be captured by global transformations.